In [1]:
pip install transformers torch datasets seqeval scikit-learn pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=a2bfe5c78e07a727636c1d00ae393b8898879013d19760d90d14965307fcdc16
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [2]:
# Install the official Kaggle API client
!pip install kaggle

In [3]:
from google.colab import files

print("Please select your 'kaggle.json' file to upload:")
files.upload()
# A file selector box will appear. Click 'Choose Files' and select the kaggle.json file.

Please select your 'kaggle.json' file to upload:


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"gangulasomashekar","key":"324fab117239ff9bc1b5060e1a2215c8"}'}

In [4]:
import os

# 1. Create the .kaggle directory if it doesn't exist
!mkdir -p ~/.kaggle

# 2. Move the uploaded kaggle.json file into the .kaggle directory
!mv kaggle.json ~/.kaggle/

# 3. Set permissions: The file must be read-only for security (owner only)
!chmod 600 ~/.kaggle/kaggle.json

print("\nKaggle API Key setup complete! You are now authenticated.")

# Verify the file is in place and permissions are correct (optional)
!ls -l ~/.kaggle/


Kaggle API Key setup complete! You are now authenticated.
total 4
-rw------- 1 root root 73 Nov 11 16:55 kaggle.json


In [5]:
import os

# 1. Define the desired folder name
DOWNLOAD_PATH = './clinical_data'

# 2. Create the folder if it doesn't exist
# The -p flag in !mkdir ensures no error is thrown if the directory already exists
!mkdir -p {DOWNLOAD_PATH}
print(f"Directory '{DOWNLOAD_PATH}' created.")

# 3. Download the dataset into the specified folder using the -p argument
# The -d flag specifies the dataset, and the -p flag specifies the path.
!kaggle datasets download -d azmayensabil/doctor-patient-conversation-large -p {DOWNLOAD_PATH}
print("Download complete.")

# 4. Unzip the downloaded file inside the target folder
# The zip file will be located at: ./clinical_data/doctor-patient-conversation-large.zip
ZIP_FILE_PATH = os.path.join(DOWNLOAD_PATH, 'doctor-patient-conversation-large.zip')

# -q for quiet (optional), -d for destination directory
!unzip -q {ZIP_FILE_PATH} -d {DOWNLOAD_PATH}
print(f"Unzip complete. Files are extracted to: {DOWNLOAD_PATH}")

# 5. (Optional) List the contents of the folder to confirm the download and unzip
print("\n--- Folder Contents ---")
!ls {DOWNLOAD_PATH}

Directory './clinical_data' created.
Dataset URL: https://www.kaggle.com/datasets/azmayensabil/doctor-patient-conversation-large
License(s): unknown
  0% 0.00/786k [00:00<?, ?B/s]
100% 786k/786k [00:00<00:00, 749MB/s]
Download complete.
Unzip complete. Files are extracted to: ./clinical_data

--- Folder Contents ---
CAR0001.txt			       RES0010.txt  RES0081.txt  RES0151.txt
CAR0002.txt			       RES0011.txt  RES0082.txt  RES0152.txt
CAR0003.txt			       RES0012.txt  RES0083.txt  RES0153.txt
CAR0004.txt			       RES0013.txt  RES0084.txt  RES0154.txt
CAR0005.txt			       RES0014.txt  RES0085.txt  RES0155.txt
DER0001.txt			       RES0015.txt  RES0086.txt  RES0156.txt
doctor-patient-conversation-large.zip  RES0016.txt  RES0087.txt  RES0158.txt
GAS0001.txt			       RES0017.txt  RES0088.txt  RES0159.txt
GAS0002.txt			       RES0018.txt  RES0089.txt  RES0160.txt
GAS0003.txt			       RES0019.txt  RES0090.txt  RES0161.txt
GAS0004.txt			       RES0020.txt  RES0091.txt  RES0162.txt
GAS0005.txt			 

In [ ]:
import os
import sys
import json
import re
import argparse
import warnings
from pathlib import Path
from typing import List, Dict, Tuple
from collections import defaultdict, Counter
from datetime import datetime
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Check imports
try:
    import torch
    from transformers import (
        AutoTokenizer,
        AutoModelForTokenClassification,
        TrainingArguments,
        Trainer,
        DataCollatorForTokenClassification,
        pipeline
    )
    from datasets import Dataset
    from sklearn.model_selection import train_test_split
    import numpy as np
    from seqeval.metrics import f1_score, precision_score, recall_score
except ImportError as e:
    print("❌ Missing dependencies. Please install:")
    print("\npip install transformers torch datasets seqeval scikit-learn pandas tqdm\n")
    sys.exit(1)

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """Project Configuration"""

    def __init__(self, data_dir: str):
        self.PROJECT_ROOT = Path.cwd()
        self.RAW_DATA_DIR = Path(data_dir)
        self.OUTPUT_DIR = self.PROJECT_ROOT / "outputs"
        self.MODEL_DIR = self.PROJECT_ROOT / "models"
        self.PROCESSED_DATA_DIR = self.PROJECT_ROOT / "processed_data"

        # Create directories
        self.OUTPUT_DIR.mkdir(exist_ok=True)
        self.MODEL_DIR.mkdir(exist_ok=True)
        self.PROCESSED_DATA_DIR.mkdir(exist_ok=True)
        (self.OUTPUT_DIR / "summaries").mkdir(exist_ok=True)
        (self.OUTPUT_DIR / "entities").mkdir(exist_ok=True)
        (self.OUTPUT_DIR / "statistics").mkdir(exist_ok=True)

        # BEST MODEL FOR CLINICAL NOTES
        self.MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

        # Training Parameters
        self.MAX_LENGTH = 512
        self.BATCH_SIZE = 16  # Reduce to 8 if GPU memory issues
        self.LEARNING_RATE = 2e-5
        self.NUM_EPOCHS = 5
        self.WEIGHT_DECAY = 0.01
        self.WARMUP_STEPS = 500

        # Entity Labels (BIO Tagging)
        self.ENTITY_LABELS = [
            'O',
            'B-SYMPTOM', 'I-SYMPTOM',
            'B-MEDICATION', 'I-MEDICATION',
            'B-TEST', 'I-TEST',
            'B-DIAGNOSIS', 'I-DIAGNOSIS',
            'B-BODY_PART', 'I-BODY_PART'
        ]

        # Data Splits
        self.TRAIN_SPLIT = 0.70
        self.VAL_SPLIT = 0.15
        self.TEST_SPLIT = 0.15

        # Options
        self.USE_POST_PROCESSING = True

# ============================================================================
# CONVERSATION PARSER
# ============================================================================

class ConversationParser:
    """Parse D:/P: format conversations"""

    def __init__(self, config):
        self.config = config

    def parse_file(self, filepath: Path) -> Dict:
        """Parse a single conversation file"""
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()

        turns = []
        for line in lines:
            line = line.strip()
            if not line:
                continue

            # Parse D: or P: format
            if line.startswith('D:'):
                speaker = 'doctor'
                text = line[2:].strip()
            elif line.startswith('P:'):
                speaker = 'patient'
                text = line[2:].strip()
            else:
                continue

            if text:
                turns.append({'speaker': speaker, 'text': text})

        return {
            'filename': filepath.name,
            'filepath': str(filepath),
            'turns': turns,
            'num_turns': len(turns)
        }

    def parse_all_files(self, data_dir: Path) -> List[Dict]:
        """Parse all conversation files"""
        files = sorted(list(data_dir.glob('*.txt')))

        if len(files) == 0:
            print(f"❌ No .txt files found in {data_dir}")
            print("Please check the directory path!")
            sys.exit(1)

        conversations = []
        print(f"📄 Parsing {len(files)} conversation files...")

        for filepath in tqdm(files, desc="Parsing"):
            try:
                conversation = self.parse_file(filepath)
                conversations.append(conversation)
            except Exception as e:
                print(f"Error parsing {filepath.name}: {e}")

        return conversations

# ============================================================================
# MEDICAL ANNOTATOR
# ============================================================================

class MedicalAnnotator:
    """Automatic annotation using comprehensive medical dictionaries"""

    def __init__(self, config):
        self.config = config
        self.entity_patterns = self._load_medical_patterns()

    def _load_medical_patterns(self) -> Dict:
        """Load comprehensive medical entity patterns"""
        return {
            'SYMPTOM': [
                'pain', 'ache', 'aching', 'sore', 'soreness', 'painful',
                'fever', 'febrile', 'temperature', 'hot',
                'cough', 'coughing', 'productive cough', 'dry cough',
                'headache', 'migraine', 'head pain',
                'nausea', 'nauseous', 'vomiting', 'vomit',
                'dizzy', 'dizziness', 'lightheaded', 'light headed', 'vertigo',
                'tired', 'fatigue', 'fatigued', 'exhausted', 'weakness', 'weak',
                'shortness of breath', 'breathless', 'difficulty breathing', 'dyspnea',
                'chest pain', 'chest tightness', 'chest pressure',
                'sore throat', 'throat pain',
                'runny nose', 'nasal congestion', 'congestion', 'stuffy nose',
                'swelling', 'swollen', 'inflammation',
                'rash', 'itchy', 'itch', 'itching',
                'bleeding', 'bruising', 'numbness', 'tingling',
                'wheezing', 'sneezing',
                'chills', 'shivering',
                'sweating', 'night sweats',
                'stiffness', 'stiff',
                'cramps', 'cramping',
                'discharge', 'phlegm', 'sputum', 'mucus',
                'hoarse', 'hoarseness',
                'palpitations', 'irregular heartbeat',
                'body aches', 'muscle pain', 'joint pain'
            ],
            'MEDICATION': [
                'aspirin', 'ibuprofen', 'acetaminophen', 'paracetamol',
                'tylenol', 'advil', 'aleve', 'motrin',
                'antibiotic', 'antibiotics', 'penicillin', 'amoxicillin',
                'azithromycin', 'augmentin',
                'inhaler', 'albuterol', 'ventolin',
                'steroid', 'prednisone', 'prednisolone',
                'insulin', 'metformin',
                'lisinopril', 'atorvastatin', 'simvastatin',
                'omeprazole', 'prilosec',
                'levothyroxine',
                'antihistamine', 'benadryl', 'zyrtec', 'claritin',
                'decongestant', 'sudafed',
                'cough syrup', 'robitussin',
                'prescription', 'medication', 'medicine', 'drug', 'pill', 'tablet'
            ],
            'TEST': [
                'blood test', 'blood work', 'lab test',
                'x-ray', 'xray', 'chest x-ray',
                'mri', 'ct scan', 'cat scan',
                'ultrasound',
                'ecg', 'ekg', 'electrocardiogram',
                'stress test',
                'biopsy',
                'screening',
                'examination', 'exam', 'physical exam',
                'culture', 'throat culture',
                'covid test', 'rapid test', 'pcr test',
                'blood pressure',
                'peak flow'
            ],
            'DIAGNOSIS': [
                'diabetes', 'diabetic',
                'hypertension', 'high blood pressure',
                'infection', 'bacterial infection', 'viral infection',
                'flu', 'influenza', 'cold', 'common cold',
                'pneumonia',
                'bronchitis',
                'asthma',
                'copd',
                'allergies', 'allergy', 'allergic',
                'sinusitis', 'sinus infection',
                'strep throat', 'strep',
                'covid', 'coronavirus',
                'arthritis',
                'gerd', 'acid reflux'
            ],
            'BODY_PART': [
                'chest', 'lung', 'lungs',
                'throat',
                'nose', 'sinus', 'sinuses',
                'head', 'neck', 'back',
                'stomach', 'abdomen',
                'heart', 'cardiac',
                'knee', 'ankle', 'shoulder', 'elbow', 'wrist',
                'joint', 'joints', 'muscle', 'muscles'
            ]
        }

    def annotate_text(self, text: str) -> Tuple[List[str], List[str]]:
        """Annotate text with BIO tags"""
        tokens = re.findall(r'\b\w+\b|[^\w\s]', text.lower())
        labels = ['O'] * len(tokens)

        text_lower = text.lower()

        for entity_type, patterns in self.entity_patterns.items():
            for pattern in sorted(patterns, key=len, reverse=True):
                pattern_lower = pattern.lower()

                start = 0
                while True:
                    idx = text_lower.find(pattern_lower, start)
                    if idx == -1:
                        break

                    token_start, token_end = self._find_token_span(
                        text_lower, tokens, idx, idx + len(pattern_lower)
                    )

                    if token_start is not None and token_end is not None:
                        if labels[token_start] == 'O':
                            labels[token_start] = f'B-{entity_type}'
                            for i in range(token_start + 1, min(token_end + 1, len(labels))):
                                if labels[i] == 'O':
                                    labels[i] = f'I-{entity_type}'

                    start = idx + 1

        return tokens, labels

    def _find_token_span(self, text: str, tokens: List[str],
                         char_start: int, char_end: int) -> Tuple[int, int]:
        """Find token indices for character span"""
        current_pos = 0
        start_token = None
        end_token = None

        for i, token in enumerate(tokens):
            token_start = text.find(token, current_pos)
            if token_start == -1:
                continue

            token_end = token_start + len(token)

            if token_start < char_end and token_end > char_start:
                if start_token is None:
                    start_token = i
                end_token = i

            current_pos = token_end

        return start_token, end_token

    def annotate_conversations(self, conversations: List[Dict]) -> List[Dict]:
        """Annotate all conversation turns"""
        annotated = []

        print("🏷️  Annotating conversations...")
        for conversation in tqdm(conversations, desc="Annotating"):
            for turn in conversation['turns']:
                tokens, labels = self.annotate_text(turn['text'])

                annotated.append({
                    'tokens': tokens,
                    'ner_tags': labels,
                    'speaker': turn['speaker'],
                    'filename': conversation['filename']
                })

        return annotated

    def get_statistics(self, annotated_data: List[Dict]) -> Dict:
        """Get annotation statistics"""
        entity_counts = Counter()
        total_entities = 0

        for item in annotated_data:
            for label in item['ner_tags']:
                if label.startswith('B-'):
                    entity_type = label[2:]
                    entity_counts[entity_type] += 1
                    total_entities += 1

        return {
            'total_entities': total_entities,
            'entity_counts': dict(entity_counts),
            'total_turns': len(annotated_data)
        }

# ============================================================================
# MODEL TRAINER
# ============================================================================

class ModelTrainer:
    """Train Bio_ClinicalBERT for NER"""

    def __init__(self, config):
        self.config = config
        self.label2id = {label: i for i, label in enumerate(config.ENTITY_LABELS)}
        self.id2label = {i: label for label, i in self.label2id.items()}

        print(f"🤖 Loading model: {config.MODEL_NAME}")
        self.tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
        self.model = AutoModelForTokenClassification.from_pretrained(
            config.MODEL_NAME,
            num_labels=len(config.ENTITY_LABELS),
            id2label=self.id2label,
            label2id=self.label2id
        )
        print("✓ Model loaded")

    def prepare_datasets(self, annotated_data):
        """Prepare train/val/test datasets"""
        # Convert labels to IDs
        for item in annotated_data:
            item['ner_tags'] = [
                self.label2id.get(label, 0) for label in item['ner_tags']
            ]

        # Split data
        train_data, temp_data = train_test_split(
            annotated_data,
            test_size=(self.config.VAL_SPLIT + self.config.TEST_SPLIT),
            random_state=42
        )
        val_data, test_data = train_test_split(
            temp_data,
            test_size=self.config.TEST_SPLIT / (self.config.VAL_SPLIT + self.config.TEST_SPLIT),
            random_state=42
        )

        print(f"\n📊 Dataset splits:")
        print(f"  Train: {len(train_data)} samples")
        print(f"  Val: {len(val_data)} samples")
        print(f"  Test: {len(test_data)} samples")

        # Create HF datasets
        train_dataset = Dataset.from_list(train_data).map(
            self._tokenize_and_align, batched=True,
            remove_columns=['tokens', 'speaker', 'filename']
        )
        val_dataset = Dataset.from_list(val_data).map(
            self._tokenize_and_align, batched=True,
            remove_columns=['tokens', 'speaker', 'filename']
        )
        test_dataset = Dataset.from_list(test_data).map(
            self._tokenize_and_align, batched=True,
            remove_columns=['tokens', 'speaker', 'filename']
        )

        return train_dataset, val_dataset, test_dataset

    def _tokenize_and_align(self, examples):
        """Tokenize and align labels"""
        tokenized = self.tokenizer(
            examples["tokens"],
            truncation=True,
            is_split_into_words=True,
            padding=False,
            max_length=self.config.MAX_LENGTH
        )

        labels = []
        for i, label in enumerate(examples["ner_tags"]):
            word_ids = tokenized.word_ids(batch_index=i)
            label_ids = []

            previous_word_idx = None
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != previous_word_idx:
                    label_ids.append(label[word_idx] if word_idx < len(label) else 0)
                else:
                    label_ids.append(label[word_idx] if word_idx < len(label) else 0)
                previous_word_idx = word_idx

            labels.append(label_ids)

        tokenized["labels"] = labels
        return tokenized

    def compute_metrics(self, eval_pred):
        """Compute evaluation metrics"""
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=2)

        true_labels = [
            [self.id2label[l] for l in label if l != -100]
            for label in labels
        ]
        true_predictions = [
            [self.id2label[p] for (p, l) in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]

        return {
            "precision": precision_score(true_labels, true_predictions),
            "recall": recall_score(true_labels, true_predictions),
            "f1": f1_score(true_labels, true_predictions)
        }

    def train(self, train_dataset, val_dataset):
        """Train the model"""
        training_args = TrainingArguments(
            output_dir=str(self.config.MODEL_DIR / "checkpoints"),
            eval_strategy="epoch",
            save_strategy="epoch",
            learning_rate=self.config.LEARNING_RATE,
            per_device_train_batch_size=self.config.BATCH_SIZE,
            per_device_eval_batch_size=self.config.BATCH_SIZE,
            num_train_epochs=self.config.NUM_EPOCHS,
            weight_decay=self.config.WEIGHT_DECAY,
            warmup_steps=self.config.WARMUP_STEPS,
            logging_steps=100,
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            push_to_hub=False,
            report_to="none"
        )

        data_collator = DataCollatorForTokenClassification(tokenizer=self.tokenizer)

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            data_collator=data_collator,
            tokenizer=self.tokenizer,
            compute_metrics=self.compute_metrics
        )

        print("\n🚀 Starting training...")
        trainer.train()

        return self.model, self.tokenizer, trainer

    def save_model(self, model, tokenizer, save_path):
        """Save trained model"""
        save_path.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"✓ Model saved to: {save_path}")

# ============================================================================
# POST-PROCESSOR & INFERENCE
# ============================================================================

class EntityPostProcessor:
    """Fix tokenization and quality issues"""

    def __init__(self):
        self.multi_word_terms = {
            'SYMPTOM': ['chest pain', 'shortness of breath', 'sore throat',
                       'runny nose', 'lightheaded', 'light headed'],
            'TEST': ['blood test', 'chest x-ray', 'ct scan', 'stress test'],
            'DIAGNOSIS': ['high blood pressure', 'sinus infection']
        }
        self.stop_entities = {'all', 'er', 'gies', 'the', 'a', 'an'}

    def clean_entities(self, entities: List[Dict]) -> List[Dict]:
        """Clean and fix entities"""
        filtered = [e for e in entities if len(e['word'].strip()) > 1 and
                   e['word'].lower() not in self.stop_entities]
        merged = self._merge_adjacent(filtered)

        for entity in merged:
            entity['word'] = entity['word'].replace('##', '').strip()

        return merged

    def _merge_adjacent(self, entities: List[Dict]) -> List[Dict]:
        """Merge adjacent entities of same type"""
        if not entities:
            return []

        sorted_entities = sorted(entities, key=lambda x: x.get('start', 0))
        merged = []
        current = sorted_entities[0].copy()

        for next_entity in sorted_entities[1:]:
            if (next_entity['entity_group'] == current['entity_group'] and
                next_entity.get('start', 0) - current.get('end', 0) <= 2):
                current['word'] += ' ' + next_entity['word']
                current['end'] = next_entity.get('end', current.get('end', 0))
                current['score'] = (current['score'] + next_entity['score']) / 2
            else:
                merged.append(current)
                current = next_entity.copy()

        merged.append(current)
        return merged


class InferenceEngine:
    """Extract entities from conversations"""

    def __init__(self, config):
        self.config = config
        self.postprocessor = EntityPostProcessor()
        self.ner_pipeline = None

    def load_model(self, model_path):
        """Load trained model"""
        print(f"📥 Loading model from: {model_path}")
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForTokenClassification.from_pretrained(model_path)

        self.ner_pipeline = pipeline(
            "ner",
            model=model,
            tokenizer=tokenizer,
            aggregation_strategy="simple",
            device=0 if torch.cuda.is_available() else -1
        )
        print("✓ Model loaded for inference")

    def extract_entities(self, conversation):
        """Extract entities from conversation"""
        all_entities = []

        for turn in conversation['turns']:
            raw_entities = self.ner_pipeline(turn['text'])

            if self.config.USE_POST_PROCESSING:
                entities = self.postprocessor.clean_entities(raw_entities)
            else:
                entities = raw_entities

            for entity in entities:
                entity['speaker'] = turn['speaker']
                entity['turn_text'] = turn['text'][:100]
                # Convert numpy.float32 score to standard float for JSON serialization
                if 'score' in entity:
                    entity['score'] = float(entity['score'])

            all_entities.extend(entities)

        return {
            'filename': conversation['filename'],
            'entities': all_entities,
            'num_entities': len(all_entities)
        }

# ============================================================================
# SOAP GENERATOR
# ============================================================================

class SOAPGenerator:
    """Generate SOAP clinical notes"""

    def __init__(self, config):
        self.config = config

    def generate_soap_note(self, extraction_result: Dict) -> str:
        """Generate SOAP note"""
        entities = extraction_result['entities']
        grouped = self._group_entities(entities)

        soap = []
        soap.append("=" * 70)
        soap.append("CLINICAL ENCOUNTER SUMMARY")
        soap.append("=" * 70)
        soap.append(f"File: {extraction_result['filename']}")
        soap.append("")

        # SUBJECTIVE
        soap.append("SUBJECTIVE (S)")
        soap.append("-" * 70)
        symptoms = [e for e in grouped.get('SYMPTOM', [])
                   if e.get('speaker') == 'patient']
        if symptoms:
            soap.append("Patient-Reported Symptoms:")
            for s in sorted(set(e['word'] for e in symptoms)):
                soap.append(f"  • {s}")
        else:
            soap.append("Patient-Reported Symptoms: [None documented]")
        soap.append("")

        # OBJECTIVE
        soap.append("OBJECTIVE (O)")
        soap.append("-" * 70)
        body_parts = grouped.get('BODY_PART', [])
        tests = grouped.get('TEST', [])
        if body_parts:
            soap.append("Physical Examination:")
            parts = sorted(set(e['word'] for e in body_parts))
            soap.append(f"  • Examined: {', '.join(parts)}")
        if tests:
            soap.append("\nDiagnostic Tests:")
            for t in sorted(set(e['word'] for e in tests)):
                soap.append(f"  • {t}")
        if not body_parts and not tests:
            soap.append("  [No physical exam or tests documented]")
        soap.append("")

        # ASSESSMENT
        soap.append("ASSESSMENT (A)")
        soap.append("-" * 70)
        diagnoses = grouped.get('DIAGNOSIS', [])
        if diagnoses:
            soap.append("Clinical Impression:")
            for d in sorted(set(e['word'] for e in diagnoses)):
                soap.append(f"  • {d}")
        else:
            soap.append("  [Assessment pending additional workup]")
        soap.append("")

        # PLAN
        soap.append("PLAN (P)")
        soap.append("-" * 70)
        medications = grouped.get('MEDICATION', [])
        if medications:
            soap.append("Medications Prescribed:")
            for m in sorted(set(e['word'] for e in medications)):
                soap.append(f"  • {m}")
        else:
            soap.append("  [No medications prescribed]")
        soap.append("")

        # STATISTICS
        soap.append("ENCOUNTER STATISTICS")
        soap.append("-" * 70)
        soap.append(f"  Total clinical entities: {extraction_result['num_entities']}")
        if grouped:
            soap.append("\n  Entity breakdown:")
            for entity_type in sorted(grouped.keys()):
                soap.append(f"    - {entity_type}: {len(grouped[entity_type])}")

        soap.append("=" * 70)
        return "\n".join(soap)

    def _group_entities(self, entities: List[Dict]) -> Dict:
        """Group entities by type"""
        grouped = defaultdict(list)
        seen = set()

        for entity in entities:
            key = (entity['word'].lower(), entity['entity_group'])
            if key not in seen:
                grouped[entity['entity_group']].append(entity)
                seen.add(key)

        return dict(grouped)

    def generate_statistics(self, results: List[Dict]) -> str:
        """Generate dataset-wide statistics"""
        stats = []
        stats.append("=" * 70)
        stats.append("DATASET-WIDE STATISTICS")
        stats.append("=" * 70)
        stats.append(f"\nTotal Conversations: {len(results)}\n")

        all_entities = defaultdict(Counter)
        for result in results:
            for entity in result['entities']:
                entity_type = entity['entity_group']
                text = entity['word'].lower()
                all_entities[entity_type][text] += 1

        for entity_type in ['SYMPTOM', 'MEDICATION', 'TEST', 'DIAGNOSIS']:
            if entity_type in all_entities:
                stats.append(f"\nTop 10 {entity_type}:")
                for text, count in all_entities[entity_type].most_common(10):
                    stats.append(f"  • {text}: {count}")

        stats.append("\n" + "=" * 70)
        return "\n".join(stats)

# ============================================================================
# MAIN PIPELINE
# ============================================================================

def main():
    """Main pipeline"""
    # parser = argparse.ArgumentParser(description='Medical Conversation SLU Pipeline')
    # parser.add_argument('--data-dir', type=str, default='/content/clinical_data',
    #                    help='Path to directory containing .txt files')
    # args = parser.parse_args()

    # Directly set data_dir for Colab execution
    data_dir = '/content/clinical_data'

    print("\n╔" + "=" * 68 + "╗")
    print("║  MEDICAL CONVERSATION SLU - COMPLETE PIPELINE                   ║")
    print("╚" + "=" * 68 + "╝")
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    # Initialize configuration
    config = Config(data_dir)
    print(f"📁 Data directory: {config.RAW_DATA_DIR}")
    print(f"💾 Outputs will be saved to: {config.OUTPUT_DIR}")
    print(f"🤖 Model: {config.MODEL_NAME}\n")

    # STEP 1: Parse
    print("=" * 70)
    print("STEP 1/5: PARSING CONVERSATIONS")
    print("=" * 70)
    parser_obj = ConversationParser(config)
    conversations = parser_obj.parse_all_files(config.RAW_DATA_DIR)
    total_turns = sum(len(c['turns']) for c in conversations)
    print(f"✓ Parsed {len(conversations)} conversations ({total_turns} turns)")

    # STEP 2: Annotate
    print("\n" + "=" * 70)
    print("STEP 2/5: AUTOMATIC ANNOTATION")
    print("=" * 70)
    annotator = MedicalAnnotator(config)
    annotated_data = annotator.annotate_conversations(conversations)
    stats = annotator.get_statistics(annotated_data)
    print(f"✓ Annotated {stats['total_entities']} entities")
    for entity_type, count in sorted(stats['entity_counts'].items()):
        print(f"  • {entity_type}: {count}")

    # STEP 3: Train
    print("\n" + "=" * 70)
    print("STEP 3/5: TRAINING MODEL (This takes ~30 minutes)")
    print("=" * 70)
    trainer = ModelTrainer(config)
    train_ds, val_ds, test_ds = trainer.prepare_datasets(annotated_data)
    model, tokenizer, trainer_obj = trainer.train(train_ds, val_ds)

    # Save model
    model_path = config.MODEL_DIR / "best_model"
    trainer.save_model(model, tokenizer, model_path)

    # Evaluate
    test_results = trainer_obj.evaluate(test_ds)
    print(f"\n{'='*70}")
    print("TEST SET RESULTS:")
    print(f"{'='*70}")
    print(f"  Precision: {test_results['eval_precision']:.4f}")
    print(f"  Recall:    {test_results['eval_recall']:.4f}")
    print(f"  F1 Score:  {test_results['eval_f1']:.4f}")
    print(f"{'='*70}")

    # STEP 4: Inference
    print("\n" + "=" * 70)
    print("STEP 4/5: ENTITY EXTRACTION")
    print("=" * 70)
    inference = InferenceEngine(config)
    inference.load_model(model_path)

    all_results = []
    print(f"Extracting entities from {len(conversations)} conversations...")
    for conversation in tqdm(conversations, desc="Extracting"):
        result = inference.extract_entities(conversation)
        all_results.append(result)

    # Save entities
    entities_file = config.OUTPUT_DIR / "entities" / "all_entities.json"
    with open(entities_file, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, indent=2)

    total_entities = sum(r['num_entities'] for r in all_results)
    print(f"✓ Extracted {total_entities} entities")

    # STEP 5: Generate SOAP notes
    print("\n" + "=" * 70)
    print("STEP 5/5: GENERATING SOAP NOTES")
    print("=" * 70)
    soap_gen = SOAPGenerator(config)

    print(f"Generating {len(all_results)} SOAP notes...")
    for result in tqdm(all_results, desc="Generating"):
        soap_note = soap_gen.generate_soap_note(result)

        filename = Path(result['filename']).stem
        output_file = config.OUTPUT_DIR / "summaries" / f"{filename}_SOAP.txt"

        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(soap_note)

    print(f"✓ Generated {len(all_results)} SOAP notes")

    # Generate statistics
    stats_text = soap_gen.generate_statistics(all_results)
    stats_file = config.OUTPUT_DIR / "statistics" / "dataset_statistics.txt"
    with open(stats_file, 'w', encoding='utf-8') as f:
        f.write(stats_text)

    # Save test results
    with open(config.OUTPUT_DIR / "statistics" / "test_results.json", 'w') as f:
        json.dump(test_results, f, indent=2)

    # Final summary
    print("\n" + "=" * 70)
    print("✓ PIPELINE COMPLETED SUCCESSFULLY!")
    print("=" * 70)
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\n📊 Results:")
    print(f"  • SOAP notes: {config.OUTPUT_DIR / 'summaries'}")
    print(f"  • Entities: {config.OUTPUT_DIR / 'entities' / 'all_entities.json'}")
    print(f"  • Statistics: {config.OUTPUT_DIR / 'statistics'}")
    print(f"  • Model: {model_path}")

    print("\n" + stats_text)

    print("\n🎉 All done! Your project is ready for submission!")

if __name__ == "__main__":
    main()